# Model Architecture Check

In [1]:
import os

os.chdir("..")

## Backbone

In [2]:
from torchvision.models.resnet import resnet50
from torchvision.models.detection.backbone_utils import BackboneWithFPN

backbone_base = resnet50(weights="DEFAULT")
backbone = BackboneWithFPN(
    backbone_base,
    return_layers={
        "layer1": "0",
        "layer2": "1",
        "layer3": "2",
        "layer4": "3",
    },
    in_channels_list=[256, 512, 1024, 2048],
    out_channels=256,
)

In [9]:
import torch

BATCH_SIZE = 5

ims = [torch.rand(3, 512, 1024) for _ in range(BATCH_SIZE)]
im_tensor = torch.stack(ims)

features = backbone(im_tensor)

In [10]:
for k, v in features.items():
    print(k)
    print(v.shape)

0
torch.Size([5, 256, 128, 256])
1
torch.Size([5, 256, 64, 128])
2
torch.Size([5, 256, 32, 64])
3
torch.Size([5, 256, 16, 32])
pool
torch.Size([5, 256, 8, 16])


## RPN

In [11]:
from torchvision.models.detection.rpn import (
    AnchorGenerator,
    RegionProposalNetwork,
    RPNHead,
)

anchor_generator = AnchorGenerator(
    sizes=((32,), (64,), (128,), (256,), (512,)),
    aspect_ratios=((0.5, 1.0, 2.0),) * 5,
)

rpn_head = RPNHead(
    in_channels=256,
    num_anchors=anchor_generator.num_anchors_per_location()[0],
)

rpn = RegionProposalNetwork(
    anchor_generator,
    rpn_head,
    fg_iou_thresh=0.7,
    bg_iou_thresh=0.3,
    batch_size_per_image=256,
    positive_fraction=0.5,
    pre_nms_top_n=dict(training=2000, testing=1000),
    post_nms_top_n=dict(training=2000, testing=1000),
    nms_thresh=0.7,
)

In [18]:
from torchvision.models.detection.image_list import ImageList

N_BBOXES = 10

il = ImageList(im_tensor, [(512, 1024) for _ in range(BATCH_SIZE)])
bboxes = [
    {
        "boxes": torch.rand(N_BBOXES, 4),
        "labels": torch.ones((N_BBOXES,), dtype=torch.int64),
    }
    for _ in range(BATCH_SIZE)
]
proposals, loss = rpn(il, features, bboxes)

In [24]:
for prop in proposals:
    print(prop.shape)

torch.Size([2000, 4])
torch.Size([2000, 4])
torch.Size([2000, 4])
torch.Size([2000, 4])
torch.Size([2000, 4])


## RoI heads

In [26]:
from torchvision.models.detection.roi_heads import RoIHeads
from torchvision.ops import MultiScaleRoIAlign
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.faster_rcnn import TwoMLPHead

box_roi_pool = MultiScaleRoIAlign(
    featmap_names=["0", "1", "2", "3"],
    output_size=7,
    sampling_ratio=2,
)

representation_size = 1024

box_head = TwoMLPHead(
    in_channels=256 * 7 * 7,
    representation_size=representation_size,
)

box_predictor = FastRCNNPredictor(
    representation_size,
    num_classes=2,
)

roi_heads = RoIHeads(
    box_roi_pool=box_roi_pool,
    box_head=box_head,
    box_predictor=box_predictor,
    fg_iou_thresh=0.5,
    bg_iou_thresh=0.5,
    batch_size_per_image=512,
    positive_fraction=0.25,
    bbox_reg_weights=None,
    score_thresh=0.05,
    nms_thresh=0.5,
    detections_per_img=100,
)

In [28]:
detections, bbox_loss = roi_heads(
    features, proposals, [(512, 1024) for _ in range(BATCH_SIZE)], bboxes
)

In [29]:
print(bbox_loss)

{'loss_classifier': tensor(0.7897, grad_fn=<NllLossBackward0>), 'loss_box_reg': tensor(0.0044, grad_fn=<DivBackward0>)}


In [33]:
for det in detections:
    print(det.shape)